# Inspect the plotting DataFrame

This notebook loads the same joined DataFrame used by `plots/app.py`.

The DataFrame contains data from `runs`, `results`, `metrics`, and `influence`. A single `result_id` therefore appears in many rows—once for its different approximators and metrics.


In [ ]:
from pathlib import Path
import sys
import plots.hessian_data as D
import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "plots" / "hessian_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "plots"))


DB_PATH = ROOT / "experiments" / "outputs" / "runs.db"
DB_PATH


PosixPath('/root/Hessian-Approximation-for-Toy-Models/experiments/outputs/runs.db')

## Load the DataFrame


In [2]:
df = D.load_runs_db(DB_PATH)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")


Loaded 438,756 rows and 37 columns


In [3]:
pd.Series(
    {
        "rows": len(df),
        "columns": len(df.columns),
        "memory_MiB": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
        "run_ids": df["run_id"].nunique(),
        "result_ids": df["result_id"].nunique(),
        "models": df["model_id"].nunique(),
        "approximators": df["approximator"].nunique(),
    },
    name="value",
).to_frame()


,value
rows,438756.0
columns,37.0
memory_MiB,338.1
run_ids,33.0
result_ids,1716.0
models,5.0
approximators,13.0


## Look at rows


In [ ]:
df.head()


## Inspect columns


In [ ]:
column_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "unique": df.nunique(dropna=True),
    }
)
column_summary


## Inspect result identities

This removes the metrics/influence repetition and shows one row per `result_id`.


In [ ]:
identity_columns = [
    "result_id",
    "run_id",
    "model_id",
    "epoch",
    "damping_value",
    "damping_strategy",
    "pseudo_target_strategy",
    "collector_subset_size",
    "analysis_seed",
    "dataset_name",
]
identity_columns = [column for column in identity_columns if column in df.columns]

result_identities = (
    df[identity_columns]
    .drop_duplicates("result_id")
    .sort_values(["model_id", "epoch", "damping_value", "result_id"])
    .reset_index(drop=True)
)
result_identities.head(50)


## Available metric combinations


In [ ]:
metric_columns = ["reference", "computation_type", "metric", "approximator"]
(
    df[metric_columns]
    .dropna(subset=["computation_type", "metric"])
    .drop_duplicates()
    .sort_values(metric_columns)
    .reset_index(drop=True)
)


## Scratch filter

Edit the conditions below for the configuration you want to inspect.


In [ ]:
filtered = df[
    (df["model_id"] == df["model_id"].dropna().iloc[0])
    # & (df["epoch"] == 10)
    # & (df["damping_value"] == 1e-3)
    # & (df["pseudo_target_strategy"] == "mcmc")
]

filtered.head(50)
